# Dataset Aggregation, Filtering & Merging


In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

In [2]:
billings      = pd.read_csv('../../data/cleaned/billings_cleaned.csv',      low_memory=False)
renewal_calls = pd.read_csv('../../data/cleaned/renewal_calls_cleaned.csv', low_memory=False)
emails        = pd.read_csv('../../data/cleaned/emails_cleaned.csv',        low_memory=False)
cc_calls      = pd.read_csv('../../data/cleaned/cc_calls_cleaned.csv',      low_memory=False)

print(f'billings:      {billings.shape}')
print(f'renewal_calls: {renewal_calls.shape}')
print(f'emails:        {emails.shape}')
print(f'cc_calls:      {cc_calls.shape}')

billings:      (113766, 35)
renewal_calls: (153269, 52)
emails:        (123385, 26)
cc_calls:      (31361, 30)


## Preprocessing

In [3]:
# converting date columns to datetime format
billings['prospect_renewal_date'] = pd.to_datetime(
    billings['prospect_renewal_date'], errors='coerce'
)
billings['registration_date'] = pd.to_datetime(
    billings['registration_date'], errors='coerce'
)
billings['closed_date'] = pd.to_datetime(
    billings['closed_date'], errors='coerce'
)

renewal_calls['call_date'] = pd.to_datetime(
    renewal_calls['call_date'], errors='coerce'
)

cc_calls['call_date'] = pd.to_datetime(
    cc_calls['call_date'], errors='coerce'
)


In [4]:
# removing residual duplicates

before = len(renewal_calls)
renewal_calls = renewal_calls.drop_duplicates()
print(f'renewal_calls: removed {before - len(renewal_calls)} exact duplicates')

before = len(cc_calls)
cc_calls = cc_calls.drop_duplicates()
print(f'cc_calls: removed {before - len(cc_calls)} exact duplicates')

renewal_calls: removed 2 exact duplicates
cc_calls: removed 7 exact duplicates


In [5]:
# converting yes/no columns to binary [1/0]
cc_binary_cols = [
    'cc_care_package_discussed', 'cc_urgency_getting_on_site',
    'cc_external_consultant', 'cc_agent_cross_sell_attempt',
    'cc_customer_issues_concerns', 'cc_business_struggles_financial_hardship',
    'cc_questionnaire_completion', 'cc_chasing_response',
    'cc_issues_within_questionnaire', 'cc_login_issues', 'cc_platform_issues',
    'cc_dissatisfaction_time_to_complete', 'cc_process_complexity_concerns',
    'cc_questions_harder_than_expected', 'cc_dissatisfaction_support',
    'cc_pricing_mentioned', 'cc_pricing_sentiment_impact',
    'cc_refund_discussed', 'cc_contractor_suggest_leave', 'cc_contractor_complained',
]

for col in cc_binary_cols:
    cc_calls[col] = (cc_calls[col].astype(str).str.strip() == 'Yes').astype(int)

print(f'Converted {len(cc_binary_cols)} Yes/No columns in cc_calls to binary.')

Converted 20 Yes/No columns in cc_calls to binary.


In [6]:
# mapping sentiment scores to numeric values
sentiment_score_map = {
    'Dissatisfied':  0,
    'Neutral':       1,
    'Satisfied':     2,
    'Not Discussed': np.nan,
    'Unknown':       np.nan,
}
emails['crm_contractor_sentiment_score_num'] = (
    emails['crm_contractor_sentiment_score']
    .astype(str).str.strip()
    .map(sentiment_score_map)
)
print('crm_contractor_sentiment_score_num value counts:')
print(emails['crm_contractor_sentiment_score_num'].value_counts(dropna=False))

crm_contractor_sentiment_score_num value counts:
crm_contractor_sentiment_score_num
1.0    53573
NaN    52228
2.0    11413
0.0     6171
Name: count, dtype: int64


---
## Billings: Keeping Only the Most Recent Record Per Customer

For churn prediction we need the **latest** record. Using an older record would mean we are predicting
on stale features (old score, old outcome). The most recent row represents the customer's
current state.

In [7]:
billings['_date_year'] = billings['prospect_renewal_date'].dt.year

billings = (
    billings
    .sort_values('prospect_renewal_date', ascending=False)
    .drop_duplicates(subset='co_ref', keep='first')
    .reset_index(drop=True)
)

billings = billings.drop(columns=['_date_year'])

print(f'renewal_year distribution:')
print(billings['renewal_year'].value_counts().sort_index())

renewal_year distribution:
renewal_year
2023     4440
2024     4014
2025    28974
2026     8898
Name: count, dtype: int64


---
## Keeping only Churned and Won Outcomes


Rows with `prospect_outcome = 'Open'` have an unknown final outcome.
So they cannot be used to train or evaluate a churn model.

In [8]:
billings = billings[billings['prospect_outcome'].isin(['Won', 'Churned'])].copy()

# create the churn target variable
billings['churn'] = (billings['prospect_outcome'] == 'Churned').astype(int)

print(f'\nBillings after removing unknown outcomes: {billings.shape}')




Billings after removing unknown outcomes: (46326, 36)


---
## Emails: Aggregating Duplicates

- Categorical Yes/No columns -> **mode** : takes the dominant signal across rows
- Numeric sentiment score -> **max** (take the most informative non-null value)
- `crm_agent_chase_count` (Low/Medium/High) -> **most severe level** seen (max priority)

In [9]:
email_yn_cols = [
    'crm_accreditation_completed', 'crm_timely_completion',
    'crm_progress_towards_accreditation', 'crm_delays_in_accreditation',
    'crm_contractor_suggested_leave', 'crm_contractor_engagement',
    'crm_dts_or_ssip_mentioned', 'crm_customer_payment_intention',
    'crm_competitors_mentioned', 'crm_platform_issues_raised',
    'crm_agent_chased_contractor', 'crm_accreditation_issues',
    'crm_membership_overdue', 'crm_dissatisified_with_renewal_price',
    'crm_customer_complained', 'crm_refund_mentioned',
    'crm_negative_customer_experience', 'crm_dissatisfaction_with_support',
    'crm_financial_hardship_mentioned',
]

# For Yes/No columns: if any one row says "Yes", the aggregated value should be "Yes"

emails_clean = emails.copy()

for col in email_yn_cols:
    emails_clean[f'{col}_bin'] = (
        emails_clean[col].astype(str).str.strip().isin(['Yes', 'yes'])
    ).astype(int)

bin_cols = [f'{c}_bin' for c in email_yn_cols]

# convert Low/Medium/High to numbers for aggregation
chase_map = {'Low': 1, 'Medium': 2, 'High': 3, 'Unknown': np.nan}
emails_clean['chase_ordinal'] = (
    emails_clean['crm_agent_chase_count']
    .astype(str).str.strip().map(chase_map)
)

# Aggregate binary flags with max
emails_agg_bin = emails_clean.groupby('co_ref')[bin_cols + ['chase_ordinal', 'crm_contractor_sentiment_score_num']].max().reset_index()

# take mode for text columns
text_cols_email = ['crm_contractor_sentiment', 'crm_membership_level', 'sentiment_category']
def safe_mode(x):
    m = x.dropna()
    m = m[~m.isin(['Unknown','Not Discussed',''])]
    if len(m) == 0:
        return np.nan
    return m.mode().iloc[0]

emails_agg_text = emails_clean.groupby('co_ref')[text_cols_email].agg(safe_mode).reset_index()

# Merge binary + text aggregations
emails_agg = emails_agg_bin.merge(emails_agg_text, on='co_ref', how='left')

# Map numbers back to label
chase_reverse = {1: 'Low', 2: 'Medium', 3: 'High'}
emails_agg['crm_agent_chase_count_agg'] = emails_agg['chase_ordinal'].map(chase_reverse)
emails_agg = emails_agg.drop(columns=['chase_ordinal'])

# Rename binary columns back to original names
rename_bin = {f'{c}_bin': c for c in email_yn_cols}
emails_agg = emails_agg.rename(columns=rename_bin)

print(f'Emails before aggregation: {emails.shape}')
print(f'Emails after aggregation:  {emails_agg.shape}')
print(f'Unique co_refs after agg:  {emails_agg["co_ref"].nunique()}')

Emails before aggregation: (123385, 27)
Emails after aggregation:  (37964, 25)
Unique co_refs after agg:  37964


---
## Renewal Calls: Apply 45-to-14 Day Window Filter

This filter needs the `prospect_renewal_date` from billings, so we merge the date
temporarily just for the filter, then work with the filtered calls.

In [10]:
renewal_dates = billings[['co_ref', 'prospect_renewal_date']]

renewal_calls_dated = renewal_calls.merge(renewal_dates, on='co_ref', how='inner')
print(f'renewal_calls after inner join with billings dates: {renewal_calls_dated.shape}')

# 45-to-14 day window
window_start = renewal_calls_dated['prospect_renewal_date'] - pd.Timedelta(days=45)
window_end   = renewal_calls_dated['prospect_renewal_date'] - pd.Timedelta(days=14)

mask = (
    (renewal_calls_dated['call_date'] >= window_start) &
    (renewal_calls_dated['call_date'] <= window_end)
)

renewal_calls_window = renewal_calls_dated[mask].copy()

# drop the temporarily added renewal date column
renewal_calls_window = renewal_calls_window.drop(columns=['prospect_renewal_date'])

print(f'\nCalls within 45-to-14 day window: {len(renewal_calls_window):,}')
print(f'Unique customers with calls in window: {renewal_calls_window["co_ref"].nunique():,}')


renewal_calls after inner join with billings dates: (151639, 53)

Calls within 45-to-14 day window: 21,956
Unique customers with calls in window: 17,803


## Renewal Calls: One Row Per Customer

We combine all calls for each customer into a single row using these rules:

- **Binary flags (0/1):** use `max`  
   If something happened even once, we keep it as 1 (e.g., a complaint).

- **Call counts / numbers:** use `sum`, `max`, or `mean`  
   Helps capture how often and how deeply the customer engaged.

- **`desire_to_cancel_clean`:** keep the worst case  
   Priority: `cancel > not_discussed > renew`.

- **`customer_response_score`:** use `min`  
   Even one bad experience matters the most.

- **`call_direction`:** count each type  
   Number of inbound vs outbound calls shows who is initiating contact.

- **Category columns (like complaint type):** use `mode` + check if it exists  
   Keep the most common category and whether it appeared at all.

In [11]:
flag_cols = [
    'serious_complaint_flag', 'other_complaint_flag', 'price_discussed_flag',
    'price_impact_renewal_flag', 'discount_requested_flag', 'call_reschedule_flag',
    'membership_alert_flag', 'agent_initiated_renewal_flag', 'competitor_mentioned_flag',
    'switching_intent_flag', 'price_switching_flag', 'pct_price_increase_flag',
    'monetary_price_increase_flag', 'customer_asked_justification_flag',
    'discount_offered_flag', 'renewal_confirmed_flag',
]

# call volume features (all rows)
call_volume = renewal_calls_window.groupby('co_ref').agg(
    total_calls           = ('call_id', 'count'),
    outbound_calls        = ('call_direction', lambda x: (x == 'Outbound').sum()),
    inbound_calls         = ('call_direction', lambda x: (x == 'Inbound').sum()),
    max_call_number       = ('call_number', 'max'),
    num_analysed_calls    = ('is_analysed', 'sum'),
).reset_index()

print(f'Call volume features: {call_volume.shape}')

Call volume features: (17803, 6)


In [12]:
# Signal features from analysed calls only
analysed = renewal_calls_window[renewal_calls_window['is_analysed'] == 1].copy()
print(f'Analysed call rows in window: {len(analysed):,}')

# Binary flags: max across all analysed calls
flag_agg = analysed.groupby('co_ref')[flag_cols].max().reset_index()
# Also get sum (how many times it happened adds more granularity)
flag_sum = analysed.groupby('co_ref')[[
    'serious_complaint_flag', 'competitor_mentioned_flag',
    'switching_intent_flag', 'discount_requested_flag'
]].sum().reset_index().rename(columns={
    'serious_complaint_flag':   'serious_complaint_count',
    'competitor_mentioned_flag':'competitor_mention_count',
    'switching_intent_flag':    'switching_intent_count',
    'discount_requested_flag':  'discount_requested_count',
})

# desire_to_cancel: take the most severe signal seen across all calls
# cancel > not_discussed > renew
desire_priority = {'cancel': 3, 'not_discussed': 2, 'renew': 1}
analysed['desire_priority'] = (
    analysed['desire_to_cancel_clean']
    .astype(str).str.strip()
    .map(desire_priority)
    .fillna(0)
)
desire_agg = (
    analysed.groupby('co_ref')['desire_priority']
    .max()
    .map({3: 'cancel', 2: 'not_discussed', 1: 'renew', 0: np.nan})
    .reset_index()
    .rename(columns={'desire_priority': 'desire_to_cancel_agg'})
)

# customer_response_score: take the minimum (most negative experience)
response_agg = (
    analysed.groupby('co_ref')['customer_response_score']
    .min()
    .reset_index()
    .rename(columns={'customer_response_score': 'min_customer_response_score'})
)
# Also take mean
response_mean = (
    analysed.groupby('co_ref')['customer_response_score']
    .mean()
    .reset_index()
    .rename(columns={'customer_response_score': 'mean_customer_response_score'})
)

# Categorical columns: mode (most common category seen across calls)
# Also create a binary flag for whether any complaint category was present
def safe_mode(x):
    m = x.dropna()
    if len(m) == 0:
        return np.nan
    return m.mode().iloc[0]

cat_rc_cols = [
    'complaint_category', 'customer_reaction_category',
    'agent_renewal_pitch_category', 'customer_renewal_response_category',
    'agent_response_category',
]
cat_agg = analysed.groupby('co_ref')[cat_rc_cols].agg(safe_mode).reset_index()

# Whether a complaint category was ever logged
cat_agg['had_complaint_category'] = cat_agg['complaint_category'].notna().astype(int)

print(f'Aggregations done for analysed calls.')

Analysed call rows in window: 12,137
Aggregations done for analysed calls.


In [13]:
# ── Merge all renewal call aggregations together ─────────────────────────────
renewal_agg = call_volume.copy()
for agg_df in [flag_agg, flag_sum, desire_agg, response_agg, response_mean, cat_agg]:
    renewal_agg = renewal_agg.merge(agg_df, on='co_ref', how='left')

# Flag: customer had at least one call in the window
renewal_agg['had_renewal_call'] = 1

print(f'Renewal calls aggregated: {renewal_agg.shape}')
print(f'Unique customers: {renewal_agg["co_ref"].nunique():,}')

Renewal calls aggregated: (17803, 36)
Unique customers: 17,803


## CC Calls: One Row Per Customer

**Aggregation rules:**

- **Binary columns (0/1):** use `max`  
   If an issue happened even once, we mark it as 1.

- **Sentiment scores (0–100):** use `mean`  
   Gives a stable average of customer sentiment.

- **`cc_contractor_sentiment` (text):** use `mode` (prioritize "Dissatisfied")  
   The most critical sentiment matters the most.

- **Call count:** use `count`  
   More calls usually indicate more customer friction.

In [14]:
# Dissatisfied > Neutral > Satisfied > Not Discussed
cc_sent_priority = {'Dissatisfied': 3, 'Neutral': 2, 'Satisfied': 1, 'Not Discussed': 0}
cc_calls['cc_sentiment_priority'] = (
    cc_calls['cc_contractor_sentiment']
    .astype(str).str.strip()
    .map(cc_sent_priority)
    .fillna(0)
)

# Numeric score columns
cc_score_cols = [
    'cc_contractor_sentiment_start_score',
    'cc_contractor_sentiment_end_score',
    'cc_contractor_sentiment_overall_score',
    'cc_contractor_sentiment_issues_score',
]

# Binary columns
cc_agg_dict = {'call_date': 'count'}  # total CC calls

for col in cc_binary_cols:
    cc_agg_dict[col] = 'max'  # 1 if ever happened

for col in cc_score_cols:
    cc_agg_dict[col] = 'mean'  # average score across all CC calls

cc_agg_dict['cc_sentiment_priority'] = 'max'  # worst sentiment seen

cc_agg = cc_calls.groupby('co_ref').agg(cc_agg_dict).reset_index()
cc_agg = cc_agg.rename(columns={'call_date': 'num_cc_calls'})

# Map sentiment back to label
cc_sent_reverse = {3: 'Dissatisfied', 2: 'Neutral', 1: 'Satisfied', 0: 'Not Discussed'}
cc_agg['cc_worst_sentiment'] = cc_agg['cc_sentiment_priority'].map(cc_sent_reverse)
cc_agg = cc_agg.drop(columns=['cc_sentiment_priority'])

# Flag: had any CC call
cc_agg['had_cc_call'] = 1

print(f'CC calls aggregated: {cc_agg.shape}')
print(f'Unique customers: {cc_agg["co_ref"].nunique():,}')

CC calls aggregated: (14925, 28)
Unique customers: 14,925


## Merge All Four Datasets

We combine all datasets into one final table for modeling.

**Merge strategy:**

- **Base:** billings  
   This contains all customers we want to predict.

- **Emails:** left join  
   Not every customer received email in the 45-14 day window.

- **Renewal calls:** left join  
   Not every customer had a call in the 45–14 day window.

- **CC calls:** left join  
   Not every customer contacted customer care.

**Handling missing values:**

After merging, some customers won’t have renewal or CC call data.

- Fill missing numeric values (counts/flags) with `0`  
- Fill missing categorical values with `'None'`

In [15]:
print(f'Billings rows before merge: {len(billings):,}')

# Start with billings as the base
merged = billings.copy()

# Inner join emails — both sides already filtered
merged = merged.merge(emails_agg, on='co_ref', how='left')
print(f'After merging emails (left):  {merged.shape}')

# Left join renewal call aggregations
merged = merged.merge(renewal_agg, on='co_ref', how='left')
print(f'After merging renewal_calls (left): {merged.shape}')

# Left join CC call aggregations
merged = merged.merge(cc_agg, on='co_ref', how='left')
print(f'After merging cc_calls (left): {merged.shape}')

Billings rows before merge: 46,326
After merging emails (left):  (46326, 60)
After merging renewal_calls (left): (46326, 95)
After merging cc_calls (left): (46326, 122)


In [16]:
# Fill NaNs introduced by left joins

# counts -> 0 (they had zero calls), flags -> 0, text -> 'No_Call'
merged['had_renewal_call'] = merged['had_renewal_call'].fillna(0).astype(int)

renewal_count_cols = [
    'total_calls', 'outbound_calls', 'inbound_calls',
    'max_call_number', 'num_analysed_calls',
    'serious_complaint_count', 'competitor_mention_count',
    'switching_intent_count', 'discount_requested_count',
]
for col in renewal_count_cols:
    if col in merged.columns:
        merged[col] = merged[col].fillna(0).astype(int)

for col in flag_cols:
    if col in merged.columns:
        merged[col] = merged[col].fillna(0).astype(int)

merged['desire_to_cancel_agg']     = merged['desire_to_cancel_agg'].fillna('No_Call')
merged['had_complaint_category']   = merged['had_complaint_category'].fillna(0).astype(int)

# Customers with no CC call:
merged['had_cc_call']  = merged['had_cc_call'].fillna(0).astype(int)
merged['num_cc_calls'] = merged['num_cc_calls'].fillna(0).astype(int)

for col in cc_binary_cols:
    if col in merged.columns:
        merged[col] = merged[col].fillna(0).astype(int)

for col in cc_score_cols:
    if col in merged.columns:
        # Fill mean score for customers with no CC calls, as they are not missing values but rather "no negative sentiment observed"
        merged[col] = merged[col].fillna(merged[col].mean())

merged['cc_worst_sentiment'] = merged['cc_worst_sentiment'].fillna('No_CC_Call')

print(f'Final merged dataset: {merged.shape}')
print(f'\nChurn rate: {merged["churn"].mean():.2%}')
print(f'Total missing values: {merged.isnull().sum().sum():,}')

Final merged dataset: (46326, 122)

Churn rate: 26.60%
Total missing values: 493,317


---
## Merged Data Description

In [17]:
# Churn distribution
print(f'\nFinal dataset shape: {merged.shape}')
print(f'Churned: {merged["churn"].sum():,} ({merged["churn"].mean():.2%})')
print(f'Retained: {(merged["churn"]==0).sum():,} ({(1-merged["churn"].mean()):.2%})')

# Coverage check: how many customers had calls/emails/cc
print(f'\nData source coverage:')
print(f'  Had renewal call (45-14 window): {merged["had_renewal_call"].sum():,} ({merged["had_renewal_call"].mean():.1%})')
print(f'  Had CC call:                     {merged["had_cc_call"].sum():,} ({merged["had_cc_call"].mean():.1%})')

# Any remaining nulls
remaining_nulls = merged.isnull().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0].sort_values(ascending=False)
if len(remaining_nulls) > 0:
    print(f'\nRemaining null columns ({len(remaining_nulls)}):')
    print(remaining_nulls.head(15).to_string())
else:
    print('\nNo remaining nulls.')


Final dataset shape: (46326, 122)
Churned: 12,321 (26.60%)
Retained: 34,005 (73.40%)

Data source coverage:
  Had renewal call (45-14 window): 17,803 (38.4%)
  Had CC call:                     14,557 (31.4%)

Remaining null columns (32):
complaint_category                    44470
customer_reaction_category            42742
mean_customer_response_score          37550
min_customer_response_score           37550
agent_response_category               37412
customer_renewal_response_category    37370
agent_renewal_pitch_category          37316
sentiment_category                    11906
crm_contractor_sentiment_score_num    11906
crm_contractor_sentiment              11904
crm_membership_level                   9513
crm_accreditation_completed            8665
crm_agent_chase_count_agg              8665
crm_financial_hardship_mentioned       8665
crm_dissatisfaction_with_support       8665


In [18]:
# Column summary
print('\nFinal column list:')
for i, col in enumerate(merged.columns, 1):
    dtype = merged[col].dtype
    nulls = merged[col].isnull().sum()
    print(f'  {i:3d}. {col:<50} {str(dtype):<10} nulls={nulls}')


Final column list:
    1. co_ref                                             object     nulls=0
    2. renewal_month                                      object     nulls=0
    3. sustainability_score                               float64    nulls=0
    4. total_renewal_score_new                            float64    nulls=0
    5. auto_renewal_score                                 int64      nulls=0
    6. status_scores                                      int64      nulls=0
    7. anchoring_score                                    float64    nulls=0
    8. tenure_scores                                      float64    nulls=0
    9. current_anchorings                                 int64      nulls=0
   10. payment_timeframe                                  float64    nulls=0
   11. registration_date                                  datetime64[ns] nulls=378
   12. proforma_account_stage                             object     nulls=0
   13. proforma_audit_status                      

---
## Save Final Dataset

In [19]:
merged.to_csv('../../data/merged/merged_final.csv', index=False)
print(f'Saved merged_final.csv')


Saved merged_final.csv
